# Bangladesh ERA5-Land / BMD: Final Pre-Manuscript Analytics

**No model retraining.** This notebook independently recomputes year-by-year LOSO skill, rainfall event contingency, annual TXx/TNn/Rx1day/Rx5day, and station-paired trade-offs from **102 original frozen held-out prediction files**. It uses the private author-processed BMD data only locally and writes aggregate science outputs.

**Needs:** the original full project root containing `research_v2/input/dataset/`, `research_rebuild/code/` and `research_rebuild/results/jobs/`. The results-only shareable ZIP deliberately excludes private BMD observations and large trained model arrays. Do not present date-shift hypotheses as official BMD corrections.

In [ ]:
from pathlib import Path
import runpy, json
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/'research_rebuild/results/jobs').exists() and (p/'research_v2/input/dataset').exists()), None)
assert ROOT is not None, 'Attach the author project root with BMD and saved LOSO predictions, then reopen this notebook there.'
RUN_DIR = ROOT/'pre_manuscript_run_20261110'
assert (RUN_DIR/'run_remaining_analytics.py').exists(), 'Missing reproducibility analysis scripts'
print('Verified full source project:', ROOT)
print('Existing frozen LOSO prediction files:', len(list((ROOT/'research_rebuild/results/jobs').glob('loso__seed42__*.npz'))))


## Run all additional scientific analyses
These are CPU-only post-processing operations. They never change the source BMD/ERA observations or re-fit a model. Year exclusions are evaluation-only, not corrected-data retraining.

In [ ]:
runpy.run_path(str(RUN_DIR/'run_remaining_analytics.py'), run_name='__main__')
runpy.run_path(str(RUN_DIR/'run_station_index_tradeoffs.py'), run_name='__main__')
assert (RUN_DIR/'new_analysis/new_analysis_manifest.json').exists()
print('PASS: All new aggregate results and figures regenerated.')


## Manuscript-ready summaries (from actual computed arrays)

In [ ]:
OUT = RUN_DIR/'new_analysis'
m = pd.read_csv(OUT/'daily_model_subset_summary.csv')
display(m[m.year_selection.eq('all_processed_years')][['target','method','stations','valid_days','mean_station_rmse']].pivot(index='target',columns='method',values='mean_station_rmse').round(4))
i = pd.read_csv(OUT/'annual_climate_index_summary_by_year_subset.csv')
display(i[(i.year_selection=='all_processed_years') & i.method.isin(['RAW','ET_LOCAL'])].pivot(index='index',columns='method',values='mean_station_index_rmse').round(3))
t = pd.read_csv(OUT/'annual_index_paired_station_tradeoffs.csv')
display(t[(t.year_selection=='all_processed_years') & t.candidate.eq('ET_LOCAL')][['index','stations','station_years','stations_better','stations_worse','relative_mean_station_rmse_gain_pct']].round(2))
events = pd.read_csv(OUT/'rainfall_event_summary_by_year_subset.csv')
display(events[(events.year_selection=='all_processed_years') & events.method.isin(['RAW','ET_LOCAL'])].pivot(index='threshold_mm',columns='method',values='csi').round(3))


In [ ]:
for fig in ['F01_yearly_transfer_skill.png','F02_annual_climate_indices.png','F03_event_skill_excluding_uncertain_years.png','F04_climate_index_paired_tradeoff.png']:
    display(Image(filename=str(OUT/fig)))


### Critical limits before submission
- BMD's historic observation-date convention and station relocation history have **not** been authenticated for all 2000–2023 dates.
- Original GEE extraction script and task receipts were not recovered; preserved raw-format ERA exports agree numerically with processed inputs, but this is not source-code provenance.
- Rx1day and R10mm trade-offs are robust on measured processed labels, **not** guaranteed to survive a future source-provenance-corrected retraining.
- Descriptive station bootstrap intervals are not independent-network inferential confidence intervals.
- Never redistribute private daily BMD observations without permission.